# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Feature Engineering Strategy:**
1. **Numeric Features:** `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`.
2. **Patterned Missingness Flags:** Since missing values follow content type structure (e.g. `feedly article` missing keyword data), we engineer explicit missingness flags: `has_word_count`, `has_search_volume`, and `has_pos_data` (since `avg_position == 0` means missing data).
3. **Categorical Handling:** One-hot encoding for `content_type` and `main_intent`.

In [1]:
# Code check: Build feature matrix X and target y
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Engineer missingness indicators
df['has_word_count'] = df['word_count'].notna().astype(int)
df['has_search_volume'] = df['search_volume'].notna().astype(int)
df['has_pos_data'] = (df['avg_position'] > 0).astype(int)

base_num_cols = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 
                 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 
                 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 
                 'engagement_rate', 'scroll_rate', 'has_word_count', 'has_search_volume', 'has_pos_data']

X = df[base_num_cols].fillna(0)
# Encode categorical features
X_cat = pd.get_dummies(df[['content_type', 'main_intent']], drop_first=True, dtype=int)
X_full = pd.concat([X, X_cat], axis=1)
y = df['is_declining_label']

print(f"Feature Matrix Shape: {X_full.shape}")
print(f"Target Vector Shape: {y.shape}")
print(f"Engineered Features List:\n{list(X_full.columns)}")

Feature Matrix Shape: (30000, 23)
Target Vector Shape: (30000,)
Engineered Features List:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'has_word_count', 'has_search_volume', 'has_pos_data', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature Name | Domain Meaning | Missing Value Handling | Available Before Prediction? |
|---|---|---|---|
| `impressions_90d` | Total organic search impressions in trailing 90 days | Fill 0 | Yes (Knowable at $T_t$) |
| `ctr` | Click-through rate percentage (0.76 = 0.76%) | Fill 0 | Yes (Knowable at $T_t$) |
| `avg_position` | Average Google search ranking position | Fill 0 + flag `has_pos_data` | Yes (Knowable at $T_t$) |
| `days_since_last_update` | Days elapsed since last content modification | Fill median + flag | Yes (Knowable at $T_t$) |
| `has_word_count` | Indicator if word count is present | 1 if present, 0 if NaN | Yes (Knowable at $T_t$) |
| `has_search_volume` | Indicator if keyword search volume is present | 1 if present, 0 if NaN | Yes (Knowable at $T_t$) |

In [2]:
# Code check: Inspect feature missingness and non-null summary
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=== FEATURE MISSINGNESS SUMMARY ===")
print(df[['search_volume', 'word_count', 'cpc', 'avg_position']].isna().sum())
print("\n=== ZERO-POSITION ROWS (AVG_POSITION == 0) ===")
print((df['avg_position'] == 0).sum())

=== FEATURE MISSINGNESS SUMMARY ===
search_volume    2468
word_count       7699
cpc              2468
avg_position        0
dtype: int64

=== ZERO-POSITION ROWS (AVG_POSITION == 0) ===
1205


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Leakage Attack 1 — Label-Derived Target Leakage (`trend_pct`):**
* **Test:** We train a DecisionTreeClassifier WITH `trend_pct` vs WITHOUT `trend_pct`.
* **Result:**
  * **Leaky Model ROC-AUC (with `trend_pct`):** **1.0000** (Perfect score — absolute proof of target leakage!).
  * **Honest Model ROC-AUC (without `trend_pct`):** **0.7192**.

**Leakage Attack 2 — Grouped Split vs Random Split (Memorization Gap):**
* **Test:** We evaluate the honest model under **GroupKFold (grouped by `client_id`)** to prevent the model from memorizing client-specific baseline traffic levels.
* **Result:** GroupKFold CV ROC-AUC is **0.6463**, demonstrating true out-of-client generalization capability.

In [3]:
# Code check: Execute leakage attack and GroupKFold evaluation
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

base_num_cols = ['search_volume', 'competition', 'cpc', 'word_count', 'impressions_90d', 
                 'clicks_90d', 'pageviews_90d', 'content_age_days', 'days_since_last_update',
                 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

X_clean = df[base_num_cols].fillna(0)
y = df['is_declining_label']
groups = df['client_id']

# 1. Leaky Model Test
X_leaky = X_clean.copy()
X_leaky['trend_pct'] = df['trend_pct'].fillna(0)
dt_leaky = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_leaky.fit(X_leaky, y)
auc_leaky = roc_auc_score(y, dt_leaky.predict_proba(X_leaky)[:, 1])

# 2. Honest Model Test
dt_honest = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_honest.fit(X_clean, y)
auc_honest = roc_auc_score(y, dt_honest.predict_proba(X_clean)[:, 1])

# 3. GroupKFold Cross-Validation by client_id
gkf = GroupKFold(n_splits=5)
scores_gkf = []
for train_idx, val_idx in gkf.split(X_clean, y, groups):
    clf = DecisionTreeClassifier(max_depth=5, random_state=42)
    clf.fit(X_clean.iloc[train_idx], y.iloc[train_idx])
    scores_gkf.append(roc_auc_score(y.iloc[val_idx], clf.predict_proba(X_clean.iloc[val_idx])[:, 1]))

print(f"Leaky Model ROC-AUC (with trend_pct): {auc_leaky:.4f}")
print(f"Honest Model ROC-AUC (without trend_pct): {auc_honest:.4f}")
print(f"GroupKFold CV ROC-AUC (grouped by client_id): {np.mean(scores_gkf):.4f}")

Leaky Model ROC-AUC (with trend_pct): 1.0000
Honest Model ROC-AUC (without trend_pct): 0.7192
GroupKFold CV ROC-AUC (grouped by client_id): 0.6463


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded Field | One-Line Reason for Exclusion |
|---|---|
| `trend_pct` | Direct numerical computation of the target label (100% target leakage). |
| `trend_direction` | String category representation of the target label (100% target leakage). |
| `impressions_last_30d` | Sub-window component directly used to calculate target `trend_pct`. |
| `impressions_prev_30d` | Sub-window component directly used to calculate target `trend_pct`. |
| `clicks_last_30d` | Sub-window component overlapping the outcome measurement window. |
| `clicks_prev_30d` | Sub-window component overlapping the outcome measurement window. |
| `content_id` | Pseudonymized item identifier — context only, not a feature. |
| `client_id` | Pseudonymized client identifier — used for GroupKFold splitting, not a feature. |

In [4]:
# Code check: Verify excluded list is completely absent from clean feature matrix X
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['has_word_count'] = df['word_count'].notna().astype(int)
df['has_search_volume'] = df['search_volume'].notna().astype(int)
df['has_pos_data'] = (df['avg_position'] > 0).astype(int)

base_num_cols = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 
                 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 
                 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 
                 'engagement_rate', 'scroll_rate', 'has_word_count', 'has_search_volume', 'has_pos_data']

X_clean = df[base_num_cols].fillna(0)
X_cat = pd.get_dummies(df[['content_type', 'main_intent']], drop_first=True, dtype=int)
X_mat = pd.concat([X_clean, X_cat], axis=1)

excluded_fields = ['trend_pct', 'trend_direction', 'impressions_last_30d', 'impressions_prev_30d', 
                   'clicks_last_30d', 'clicks_prev_30d', 'content_id', 'client_id']

overlap_count = len(set(X_mat.columns).intersection(set(excluded_fields)))
print(f"Total features in final matrix: {len(X_mat.columns)}")
print(f"Excluded fields present in feature matrix count: {overlap_count}")
assert overlap_count == 0, 'Exclusion assertion failed!'

Total features in final matrix: 23
Excluded fields present in feature matrix count: 0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.